# 30 — Does support for arming Israel vary with the 2020→24 swing? (pre-registration addendum 4)

Runs the check fixed in `report/preregistration_2026_general_addendum_4.md` (committed 28 Sept 2026 before this notebook was run). CES 2024 `CC24_308b` (tick all that apply; option 3 *Provide arms to Israel*, option 1 *Do not get involved*); ANES 2024 `V241407` (sides with Israelis). County swing = Trump two-party share 2024 − 2020 (MIT county file), per 10 points. Reading rule (model 2): |change| < 3 points per 10 points of swing with interval inside ±5 → no association; clearly negative → a positive El-Sayed slope may partly reflect his Gaza position; clearly positive → the position would push the slope down.

In [1]:
import numpy as np, pandas as pd, statsmodels.formula.api as smf, warnings
warnings.filterwarnings('ignore'); pd.set_option('display.width', 220)
RAW = '../data/raw/'
c = pd.read_csv(RAW + 'ces/CCES24_Common_OUTPUT_vv_topost_final.csv', usecols=['countyfips', 'inputstate', 'pid7', 'commonweight', 'commonpostweight', 'presvote20post', 'CC24_410', 'CC24_308b_1', 'CC24_308b_3'], low_memory=False)
print('CC24_308b_3 values:', c.CC24_308b_3.value_counts(dropna=False).to_dict())
c['arms_israel'] = (c.CC24_308b_3 == 1).astype(float).where(c.CC24_308b_3.isin([1, 2]))
c['not_involved'] = (c.CC24_308b_1 == 1).astype(float).where(c.CC24_308b_1.isin([1, 2]))
cr = pd.read_csv(RAW + 'mit_election_lab/county_returns.csv')
p = cr.pivot_table(index='county_fips', columns='year', values=['dem_votes', 'rep_votes'])
sw = (100 * (p[('rep_votes', 2024)] / (p[('rep_votes', 2024)] + p[('dem_votes', 2024)]) - p[('rep_votes', 2020)] / (p[('rep_votes', 2020)] + p[('dem_votes', 2020)]))).rename('swing')
c = c.merge(sw.reset_index().rename(columns={'county_fips': 'countyfips'}), on='countyfips', how='left')
c['swing10'] = c.swing / 10; c['pid'] = c.pid7.where(c.pid7 <= 7)
print('respondents with county swing:', int(c.swing.notna().sum()), 'of', len(c))

CC24_308b_3 values: {2: 44272, 1: 15728}
respondents with county swing: 59921 of 60000


## Place level (primary)

In [2]:
rows = []
for y in ['arms_israel', 'not_involved']:
    for lab, df, rhs in [('1 all', c, 'swing10'), ('2 + party ID', c, 'swing10 + pid'), ('3 Michigan', c[c.inputstate == 26], 'swing10 + pid')]:
        x = df.dropna(subset=[y, 'swing10', 'commonweight'] + (['pid'] if 'pid' in rhs else []))
        r = smf.wls(f'{y} ~ {rhs}', data=x, weights=x.commonweight).fit(cov_type='cluster', cov_kwds={'groups': x.countyfips})
        ci = r.conf_int().loc['swing10'] * 100
        rows.append({'outcome': y, 'model': lab, 'N': len(x), 'counties': x.countyfips.nunique(), 'mean_pct': 100 * np.average(x[y], weights=x.commonweight),
                     'pts_per_10pt_swing': 100 * r.params['swing10'], 'lo': ci[0], 'hi': ci[1]})
place = pd.DataFrame(rows); print(place.round(2).to_string(index=False))
place.to_csv('../outputs/tables/israel_views_by_county_swing.csv', index=False)
m2 = place[(place.outcome == 'arms_israel') & (place.model == '2 + party ID')].iloc[0]
verdict = ('no association' if abs(m2.pts_per_10pt_swing) < 3 and m2.lo > -5 and m2.hi < 5 else
           'less support in swing places' if m2.hi < 0 else 'more support in swing places' if m2.lo > 0 else 'inconclusive')
print('\nPre-registered reading (model 2, arms to Israel):', verdict)

     outcome        model     N  counties  mean_pct  pts_per_10pt_swing     lo    hi
 arms_israel        1 all 59921      2686     26.18               -7.33 -10.89 -3.76
 arms_israel 2 + party ID 58660      2675     26.90               -1.86  -4.83  1.11
 arms_israel   3 Michigan  1907        81     26.48              -17.61 -40.76  5.54
not_involved        1 all 59921      2686     22.18                1.48  -2.22  5.18
not_involved 2 + party ID 58660      2675     21.28                2.43  -1.08  5.95
not_involved   3 Michigan  1907        81     20.79               25.83   3.53 48.14

Pre-registered reading (model 2, arms to Israel): no association


## Individual level (secondary)

In [3]:
c['v20'] = c.presvote20post.map({1: 'Biden', 2: 'Trump', 6: 'nonvoter'}); c['v24'] = c.CC24_410.map({1: 'Harris', 2: 'Trump'})
c['path'] = np.select([(c.v20 == 'Biden') & (c.v24 == 'Harris'), (c.v20 == 'Biden') & (c.v24 == 'Trump'), (c.v20 == 'nonvoter') & (c.v24 == 'Trump'), (c.v20 == 'Trump') & (c.v24 == 'Trump')],
                      ['loyal Democrats', 'Biden -> Trump (recalled)', 'new -> Trump', 'loyal Republicans'], None)
w = c.commonpostweight.fillna(c.commonweight)
ces_paths = c.dropna(subset=['path', 'arms_israel']).assign(w=w).groupby('path').apply(lambda s: pd.Series({'N': len(s), 'arms_israel_pct': 100 * np.average(s.arms_israel, weights=s.w), 'not_involved_pct': 100 * np.average(s.not_involved.fillna(0), weights=s.w)}))
print('CES 2024:'); print(ces_paths.round(1))
a = pd.read_csv(RAW + 'anes/anes_2024_core.csv', low_memory=False); a = a[a.weight_post > 0]
a['sides_israel'] = (a.israel_side == 1).astype(float).where(a.israel_side.isin([1, 2, 3, 4]))
a['path'] = np.select([(a.vote20 == 'D') & (a.vote24 == 'D'), (a.vote20 == 'D') & (a.vote24 == 'R'), (a.vote20 == 'NV') & (a.vote24 == 'R'), (a.vote20 == 'R') & (a.vote24 == 'R')],
                      ['loyal Democrats', 'Biden -> Trump (recalled)', 'new -> Trump', 'loyal Republicans'], None)
anes_paths = a.dropna(subset=['path', 'sides_israel']).groupby('path').apply(lambda s: pd.Series({'N': len(s), 'sides_israel_pct': 100 * np.average(s.sides_israel, weights=s.weight_post)}))
print('\nANES 2024:'); print(anes_paths.round(1))
ces_paths.join(anes_paths, rsuffix='_anes').to_csv('../outputs/tables/israel_views_by_path.csv')

CES 2024:
                                 N  arms_israel_pct  not_involved_pct
path                                                                 
Biden -> Trump (recalled)    632.0             20.9              27.7
loyal Democrats            20733.0             15.5              10.8
loyal Republicans          15977.0             50.6              21.8
new -> Trump                1532.0             26.2              36.4

ANES 2024:
                                N  sides_israel_pct
path                                               
Biden -> Trump (recalled)    64.0              25.2
loyal Democrats            1868.0              15.1
loyal Republicans          1361.0              60.4
new -> Trump                187.0              39.7
